In [1]:
import bw2data, bw2io
import bw2calc
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import os
import sys

In [2]:
sys.path.append('/Users/susierwu/dpLCA_main/') 
from utils import *
from utils.newbw2method_dpLCIA_addBW25 import * 

In [3]:
bw2data.projects.set_current('ei311')
#list(bw2data.databases)
#[m for m in bw2data.methods if 'pGWP100' in str(m) and  'SSP119' in str(m)]

In [4]:
#[m for m in bw2data.methods if 'pGWP100' in str(m)]

In [5]:
mybio = bw2data.Database("ecoinvent-3.11-biosphere")
len(mybio)

9795

### read in pre-calculated .nc LCIA dataset, d-IRF for a new IRF metric calculation

In [6]:
cf_point = xr.open_dataset('../../../dpLCIA/LCIA/data/CF_GWP1_100_perSSP_MY_majorghgs.nc')
cf_point

<xarray.Dataset> Size: 24kB
Dimensions:    (SSP: 3, ModelYear: 4, Year: 100)
Coordinates:
  * SSP        (SSP) object 24B '119' '245' '585'
  * ModelYear  (ModelYear) int32 16B 2020 2030 2040 2050
  * Year       (Year) int32 400B 1 2 3 4 5 6 7 8 9 ... 93 94 95 96 97 98 99 100
Data variables:
    CO2_GWP    (SSP, ModelYear, Year) int32 5kB ...
    CH4_GWP    (SSP, ModelYear, Year) float64 10kB ...
    N2O_GWP    (SSP, ModelYear, Year) float64 10kB ...

### read in premise_GWP, all minor_ghg using static amount 

In [7]:
prem_gwp_dfraw = pd.read_excel("../../../dpLCIA/LCIA/premise_gwp/lcia_gwp2021_100a_w_bio.xlsx")
prem_gwp_dfraw.head()

,name,categories,amount
0,Bromopropane,air::unspecified,0.052
1,Butane,air::urban air close to ground,0.006
2,Butane,air::non-urban air or from high stacks,0.006
3,Butane,"air::low population density, long-term",0.006
4,Butane,air::lower stratosphere + upper troposphere,0.006


### calling the assign_dpGWP class, see what it looks like for final CF

In [8]:
xx = assign_dpGWP(premise_gwp100_inputdf = prem_gwp_dfraw, cf_inputds = cf_point, ssp = '119', fairMY = 2030 )
minorg, allg = xx.get_minorand_allGHG()
cc = xx.prep_empty_C(allg)
#cc.head()
pcc = xx.assign_minorghg_to_C_GWP100(cc, minorg)
fcc = xx.assign_majorghg_dCC(pcc)
fcc

,Bromopropane,Butane,"Carbon monoxide, fossil","Carbon monoxide, from soil or biomass stock","Carbon monoxide, non-fossil",Chloroform,Ethane,"Ethane, 1,1,1,2-tetrafluoro-, HFC-134a","Ethane, 1,1,1-trichloro-, HCFC-140","Ethane, 1,1,1-trifluoro-, HFC-143a",...,"Carbon dioxide, non-fossil","Carbon dioxide, in air","Carbon dioxide, to soil or biomass stock","Carbon dioxide, from soil or biomass stock","Carbon dioxide, fossil","Carbon dioxide, non-fossil, resource correction","Methane, from soil or biomass stock","Methane, fossil","Methane, non-fossil",Dinitrogen monoxide
GWP1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,-1,-1,1,1,-1,112.507191,112.507191,112.507191,160.358301
GWP2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,-1,-1,1,1,-1,111.315029,111.315029,111.315029,164.764098
GWP3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,-1,-1,1,1,-1,109.927564,109.927564,109.927564,168.834425
GWP4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,-1,-1,1,1,-1,108.342704,108.342704,108.342704,172.564863
GWP5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,-1,-1,1,1,-1,106.595296,106.595296,106.595296,175.968976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GWP96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,-1,-1,1,1,-1,25.722710,25.722710,25.722710,193.270201
GWP97,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,-1,-1,1,1,-1,25.500141,25.500141,25.500141,192.849542
GWP98,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,-1,-1,1,1,-1,25.281528,25.281528,25.281528,192.426442
GWP99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,-1,-1,1,1,-1,25.066762,25.066762,25.066762,192.000999


### run MY2030/2040&2050 and all SSP together, for GWP we only have pGWP100 now, using static GWP100 for minorGHGs

In [10]:
for mmy in [2030, 2040, 2050]: 
    for sp in ['119', '245', '585']: 
        xx = assign_dpGWP(premise_gwp100_inputdf = prem_gwp_dfraw, cf_inputds = cf_point, ssp = sp, fairMY = mmy, GWP100_only = True )
        minorg, allg = xx.get_minorand_allGHG()
        emt_C =  xx.prep_empty_C(allg)
        fullminor_C = xx.assign_minorghg_to_C_GWP100(emt_C, minorg )
        #print(fullminor_C.head() )
        full_allC = xx.assign_majorghg_dCC(fullminor_C)
        data = xx.prep_data_for_bw2method (full_allC, 
                                           mybio = bw2data.Database("ecoinvent-3.11-biosphere") , 
                                           mybio_str = "ecoinvent-3.11-biosphere")
        print(len(data))
        xx.prep_final_dCC_bw2method (data, gwp_method = '- dp-AGWPCO2')

start preparing data to be assigned as new bw2data.methods, for SSP 119 and fair_MY2030
1
 start creating new method, with name: IPCC 2021 - SSP119-MY2030
finishing preparing new methods, method name : ('Climate Change prospective GWP100', 'SSP119', 'MY2030', 'pGWP100 - dp-AGWPCO2')
start preparing data to be assigned as new bw2data.methods, for SSP 245 and fair_MY2030
1
 start creating new method, with name: IPCC 2021 - SSP245-MY2030
finishing preparing new methods, method name : ('Climate Change prospective GWP100', 'SSP245', 'MY2030', 'pGWP100 - dp-AGWPCO2')
start preparing data to be assigned as new bw2data.methods, for SSP 585 and fair_MY2030
1
 start creating new method, with name: IPCC 2021 - SSP585-MY2030
finishing preparing new methods, method name : ('Climate Change prospective GWP100', 'SSP585', 'MY2030', 'pGWP100 - dp-AGWPCO2')
start preparing data to be assigned as new bw2data.methods, for SSP 119 and fair_MY2040
1
 start creating new method, with name: IPCC 2021 - SSP119-

In [11]:
[m for m in bw2data.methods if 'dp-AGWPCO2' in str(m)]

[('Climate Change prospective GWP100',
  'SSP119',
  'MY2030',
  'pGWP100 - dp-AGWPCO2'),
 ('Climate Change prospective GWP100',
  'SSP245',
  'MY2030',
  'pGWP100 - dp-AGWPCO2'),
 ('Climate Change prospective GWP100',
  'SSP585',
  'MY2030',
  'pGWP100 - dp-AGWPCO2'),
 ('Climate Change prospective GWP100',
  'SSP119',
  'MY2040',
  'pGWP100 - dp-AGWPCO2'),
 ('Climate Change prospective GWP100',
  'SSP245',
  'MY2040',
  'pGWP100 - dp-AGWPCO2'),
 ('Climate Change prospective GWP100',
  'SSP585',
  'MY2040',
  'pGWP100 - dp-AGWPCO2'),
 ('Climate Change prospective GWP100',
  'SSP119',
  'MY2050',
  'pGWP100 - dp-AGWPCO2'),
 ('Climate Change prospective GWP100',
  'SSP245',
  'MY2050',
  'pGWP100 - dp-AGWPCO2'),
 ('Climate Change prospective GWP100',
  'SSP585',
  'MY2050',
  'pGWP100 - dp-AGWPCO2')]